In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import umap

# QUESTION 1: Boxen Plot
df_fraud = pd.read_csv('credit_card_fraud_2026.csv')
df_fraud['log_amount'] = np.log10(df_fraud['amount_usd'])

median_order = df_fraud.groupby('device_type')['log_amount'].median().sort_values(ascending=False).index
plt.figure(figsize=(10, 6))
sns.boxenplot(data=df_fraud, x='device_type', y='log_amount', order=median_order, hue='device_type', palette='husl')
plt.title('Transaction Amount (log10 USD) by Device Type')
plt.xlabel('Device Type')
plt.ylabel('Log10 Amount (USD)')
plt.savefig('q1_boxenplot.png', bbox_inches='tight')
plt.show()

# QUESTION 2A: KDE Plot
plt.figure(figsize=(10, 6))
sns.kdeplot(data=df_fraud, x='merchant_risk_score', hue='merchant_category')
plt.title('Merchant Risk Score Density by Category')
plt.savefig('q2a_kde.png', bbox_inches='tight')
plt.show()

# QUESTION 2B: Dual KDE Plots
df_fraud['foreign_status'] = df_fraud['is_foreign_transaction'].map({False: 'Domestic', True: 'Foreign'})
df_fraud['ai_scam_status'] = df_fraud['is_ai_generated_scam_attempt'].map({False: 'Not Flagged', True: 'AI-Generated Scam'})

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.kdeplot(data=df_fraud, x='velocity_score', hue='foreign_status', ax=axes[0])
axes[0].set_title('Velocity Score by Foreign Status')
sns.kdeplot(data=df_fraud, x='velocity_score', hue='ai_scam_status', ax=axes[1])
axes[1].set_title('Velocity Score by AI Scam Status')
plt.savefig('q2b_dual_kde.png', bbox_inches='tight')
plt.show()

# QUESTION 3: Facet Grid
grouped = df_fraud.groupby(['channel', 'merchant_category'])['is_fraud'].mean().reset_index()
cat_order = df_fraud['merchant_category'].value_counts().index

g = sns.FacetGrid(grouped, col='channel', col_wrap=3, height=3.2, aspect=1.1)
g.map_dataframe(sns.barplot, x='is_fraud', y='merchant_category', order=cat_order, color='steelblue')
g.set_axis_labels('Fraud Rate', '')
g.set_titles('{col_name}')
g.fig.suptitle('Fraud Rate by Channel and Merchant Category', y=1.05)
plt.savefig('q3_facetgrid.png', bbox_inches='tight')
plt.show()

In [ ]:
# QUESTION 4: UMAP
# Run UMAP on 5,000 points
n_samples_5k = 5000
idx_5k = rng.choice(len(X_train), size=n_samples_5k, replace=False)
X_5k = X_train[idx_5k].reshape(n_samples_5k, -1)
y_5k = y_train[idx_5k]

reducer = umap.UMAP(random_state=42)
embedding_5k = reducer.fit_transform(X_5k)
plt.figure(figsize=(10, 8))
sns.scatterplot(x=embedding_5k[:, 0], y=embedding_5k[:, 1], hue=[CLASS_NAMES[i] for i in y_5k], palette='tab10', s=10)
plt.title('UMAP Projection of Fashion MNIST (5000 samples)')
plt.savefig('q4_umap_5k.png', bbox_inches='tight')
plt.show()

# Run UMAP on 60,000 points (using the full dataset subset logic)
# Ensure you allocate sufficient memory/time for 60k points
embedding_60k = reducer.fit_transform(df[pixel_cols])
plt.figure(figsize=(10, 8))
sns.scatterplot(x=embedding_60k[:, 0], y=embedding_60k[:, 1], hue=df['category'], palette='tab10', s=5)
plt.title('UMAP Projection of Fashion MNIST (60000 samples)')
plt.savefig('q4_umap_60k.png', bbox_inches='tight')
plt.show()

# QUESTION 5: Visualize with Heatmap
mean_imgs = df.groupby('label')[pixel_cols].mean()

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
axes = axes.flatten()

for class_id in range(10):
    img_grid = mean_imgs.loc[class_id].to_numpy().reshape(28, 28)
    sns.heatmap(img_grid, cmap='viridis', cbar=False, xticklabels=False, yticklabels=False, ax=axes[class_id])
    axes[class_id].set_title(CLASS_NAMES[class_id])

plt.suptitle('Average Image per Category (60000 samples)')
plt.savefig('q5_heatmap.png', bbox_inches='tight')
plt.show()

# QUESTION 6: Boxplot: Average Brightness
df['avg_brightness'] = df[pixel_cols].mean(axis=1)

# Find top 3 and lowest 2 classes based on median/mean brightness distribution
brightness_order = df.groupby('category')['avg_brightness'].median().sort_values(ascending=False).index

plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x='category', y='avg_brightness', order=brightness_order, palette='Set2')
plt.title('Average Pixel Brightness by Class Category')
plt.xticks(rotation=45)
plt.savefig('q6_boxplot.png', bbox_inches='tight')
plt.show()